# 4. Dashboard visuals (pipeline step 9)

This notebook runs **pipeline step 9** (`9_dashboard_visuals`). It **prebuilds all dashboard visualization artifacts** (BupaR, DTW, FP-Growth) on **EC2** and **saves them to S3** for **direct dashboard integration**. The dashboard loads these prebuilt assets from S3 (the API returns only URLs; no computation at request time). Visuals are **SHAP/FFA-driven**: model data and feature lists come from Step 3b / 7 / 8 so process mining and itemset mining use only important features.

**Flow:** Run after [3_model_train_shap_ffa.ipynb](3_model_train_shap_ffa.ipynb). Then run [5_build_and_deploy.ipynb](5_build_and_deploy.ipynb) once to build and deploy.

**Prerequisites:**
- **Primary**: Step 3b feature importance (`3a_feature_importance/outputs/{cohort}/{age_band}/cohort_feature_importance.csv`)
- **Fallback**: Notebook 3 combined importance (`10_risk_dashboard/outputs/{cohort}/{age_band}/combined_importance.csv`)
- If neither is available, allowed_codes will be empty and visualizations may fail or use all codes.

## Steps

1. **Setup** – Resolve paths (scripts in `9_dashboard_visuals/`; outputs under `10_risk_dashboard/visualizations/{bupar,dtw,fpgrowth}/`).
2. **BupaR** – Process mining sequences and plots (SHAP/FFA allowed codes when available); **uploaded to the dashboard bucket** under `{S3_DASHBOARD_PREFIX}/bupar/{cohort}/{age_band}/plots/`.
3. **DTW** – Trajectory features and plots **based on SHAP/FFA important codes** (same as BupaR/FP-Growth); plot PNGs are **uploaded to the dashboard bucket** under `{S3_DASHBOARD_PREFIX}/dtw/{cohort}/{age_band}/plots/`. The DTW tab includes **appointments vs no appointments** visuals: **Routine vs No Routine (Outcomes)** and **High-Risk vs Low-Risk Trajectories** (outcome rate by trajectory intensity / archetype). These visuals use the **full pipeline (2016–2019)**: model_events (Step 4) and DTW features are built from all years 2016–2019, not a single year. **Extreme-density cohorts** (optional) support the same routine vs no routine comparison for high-utilizer subgroups—see optional step below.

4. **FP-Growth** – Itemsets, rules, **Plotly network HTML**, and PNGs; **uploaded to the dashboard bucket** (`S3_DASHBOARD_BUCKET`, e.g. jerome-dixon.io) under `{S3_DASHBOARD_PREFIX}/fpgrowth/{cohort}/{age_band}/plots/` (e.g. `vcu/pgx-risk-calculator/fpgrowth/...`). The dashboard loads the **network plot by cohort** from these URLs.Idempotent. Run from repo root. Prerequisites: notebook 5 done (`4_model_data`, `7_shap_analysis`, `8_ffa_analysis`); R and bupaR for BupaR.

5. **Model performance metrics and cohort metadata** – Prebuilt via `generate_metrics.py` and `generate_metadata.py` (no recomputation). Deploy (5_build_and_deploy) uploads to the dashboard bucket: `metadata/model_performance_metrics.json` (Documentation tab) and `metadata/opioid_ed.json`, `metadata/non_opioid_ed.json` (dropdowns). Frontend loads these same-origin; Lambda GET /metrics and GET /metadata are fallbacks.
6. **API** – Returns URLs to prebuilt S3 assets only (no server-side computation for visuals).

In [ ]:
# Setup: paths (outputs under 10_risk_dashboard/visualizations/)
import sys
import os
import subprocess
from pathlib import Path

REPO_ROOT = Path.cwd()
if (REPO_ROOT / "py_helpers").exists():
    pass  # already repo root
else:
    for p in REPO_ROOT.parents:
        if (p / "py_helpers").exists():
            REPO_ROOT = p
            break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from py_helpers.env_utils import get_data_root, get_model_data_root

# Data root (with fallback to local nvme or Windows pgx_data)
DATA_ROOT = get_data_root()
MODEL_DATA_ROOT = get_model_data_root()
S3_BUCKET = os.environ.get("PGX_S3_BUCKET", "pgxdatalake")

# Creation code (step 9); outputs go to 10_risk_dashboard/visualizations
STEP9_ROOT = REPO_ROOT / "9_dashboard_visuals"
VISUAL_ROOT = REPO_ROOT / "10_risk_dashboard" / "visualizations"
BUPAR_VISUALS_SCRIPT = STEP9_ROOT / "bupar" / "create_bupar_visuals.py"
DTW_TRAJECTORIES_SCRIPT = STEP9_ROOT / "dtw" / "create_dtw_trajectories.py"
DTW_VISUALS_SCRIPT = STEP9_ROOT / "dtw" / "create_dtw_visuals.py"
FPGROWTH_VISUALS_SCRIPT = STEP9_ROOT / "fpgrowth" / "create_fpgrowth_visuals.py"

print(f"Repo root: {REPO_ROOT}")
print(f"Data root (NVMe/local): {DATA_ROOT}")
print(f"Model data root: {MODEL_DATA_ROOT}")
print(f"S3 bucket: {S3_BUCKET}")
print(f"Step 9 (scripts): {STEP9_ROOT}")
print(f"Outputs: {VISUAL_ROOT}")
print(f"Plots (PNG/HTML) appear under: {VISUAL_ROOT}/bupar/outputs/<cohort>/<age_band>/plots/ (and dtw/, fpgrowth/) — run the cells below to generate them.")

## Config: cohorts and age bands

Defaults match **run_dashboard_visuals.py**: all cohorts and all age bands (from REQUIRED_COHORTS), one worker per (cohort, age_band) combo (capped by CPU), no dry run. Leave `COHORTS_TO_RUN` and `AGE_BANDS_TO_RUN` empty for full pipeline; set either to limit scope.

In [ ]:
from py_helpers.constants import COHORT_NAMES, AGE_BANDS

try:
    from py_helpers.constants import REQUIRED_COHORTS
except ImportError:
    _all_bands = ['0-12', '13-24', '25-44', '45-54', '55-64', '65-74', '75-84', '85-114']
    REQUIRED_COHORTS = {"opioid_ed": _all_bands, "non_opioid_ed": _all_bands}

COHORTS_TO_RUN = []
AGE_BANDS_TO_RUN = []

# Default: pipeline (cohort, age_band) from REQUIRED_COHORTS (both cohorts use full age bands 0-12 through 85-114)
if not COHORTS_TO_RUN and not AGE_BANDS_TO_RUN:
    combinations = [(c, ab) for c, bands in REQUIRED_COHORTS.items() for ab in bands]
    print("Using pipeline-supported cohort/age_band (REQUIRED_COHORTS)")
else:
    if not COHORTS_TO_RUN:
        COHORTS_TO_RUN = COHORT_NAMES.copy()
    if not AGE_BANDS_TO_RUN:
        AGE_BANDS_TO_RUN = AGE_BANDS.copy()
    combinations = [(c, ab) for c in COHORTS_TO_RUN for ab in AGE_BANDS_TO_RUN]

print(f"Cohorts: {COHORTS_TO_RUN if COHORTS_TO_RUN else list(REQUIRED_COHORTS.keys())}")
print(f"Age bands: {AGE_BANDS_TO_RUN if AGE_BANDS_TO_RUN else 'per-cohort (REQUIRED_COHORTS)'}")
print(f"Total: {len(combinations)} combinations")

# Idempotent: skip when output exists. Set FORCE_RERUN=True to pass --force and re-run all (BupaR, FP-Growth, DTW).
# FP-Growth: --force re-creates itemsets and plots. DTW: --force ignores pipeline checkpoint and plots.
FORCE_RERUN = False
# Parallel workers: one per (cohort, age_band) combo, capped by CPU (matches run_dashboard_visuals.py default).
_ncpu = getattr(os, "cpu_count", lambda: 4)() or 4
PARALLEL_WORKERS = min(_ncpu, len(combinations))
FPGROWTH_WORKERS = min(_ncpu, len(combinations))

# Prerequisite: SHAP/FFA combined allowed codes (required by BupaR and DTW; we never use all codes).
# These are built from either Step 3b feature importance or notebook 3 combined_importance.csv (fallback).
import json
BUPAR_OUTPUTS = REPO_ROOT / "10_risk_dashboard" / "visualizations" / "bupar" / "outputs"
missing = []
empty = []
for cohort_name, age_band in combinations:
    age_band_fname = age_band.replace("-", "_")
    path = BUPAR_OUTPUTS / f"allowed_codes_shap_ffa_{cohort_name}_{age_band_fname}.json"
    if not path.exists():
        missing.append(f"{cohort_name}/{age_band} ({path.name})")
    else:
        try:
            with open(path, encoding="utf-8") as f:
                codes = json.load(f)
            if not codes or (isinstance(codes, list) and len(codes) == 0):
                empty.append(f"{cohort_name}/{age_band} ({path.name})")
        except Exception as e:
            empty.append(f"{cohort_name}/{age_band} ({path.name}): {e}")
if missing or empty:
    msg = "SHAP/FFA combined allowed codes are required for BupaR and DTW (prerequisite).\n"
    if missing:
        msg += f"  Missing: {', '.join(missing)}\n"
    if empty:
        msg += f"  Empty or invalid: {', '.join(empty)}\n"
    msg += "  Sources (checked in order):\n"
    msg += "    1. Step 3b feature importance: 3a_feature_importance/outputs/{cohort}/{age_band}/cohort_feature_importance.csv\n"
    msg += "    2. Notebook 3 combined importance: 10_risk_dashboard/outputs/{cohort}/{age_band}/combined_importance.csv\n"
    msg += "  Generate allowed codes by running create_bupar_visuals.py with --write-allowed-codes, or sync from S3 gold/bupar/.\n"
    msg += "  If both sources above are missing, re-run feature importance (notebook 2) or SHAP/FFA combine (notebook 3)."
    raise RuntimeError(msg)
print(f"Prerequisite check passed: all {len(combinations)} SHAP/FFA combined allowed codes files present.")
print("  Sources: Step 3b feature importance (primary) or notebook 3 combined_importance.csv (fallback)")

In [ ]:
# (Prerequisite check runs in the Config cell above.)

## Run BupaR process mining

Plots are uploaded to the dashboard bucket under `{S3_DASHBOARD_PREFIX}/bupar/{cohort}/{age_band}/plots/` (same pattern as FP-Growth). **BupaR features are not added to model data** (same as DTW and FP-Growth); they are for dashboard visualization only.

In [ ]:
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

FAIL_FAST = True
FORCE_RERUN = True
force_flag = ["--force"] if FORCE_RERUN else []

def run_bupar_one(cohort_name, age_band):
    r = subprocess.run(
        [sys.executable, str(BUPAR_VISUALS_SCRIPT), "--cohort-name", cohort_name, "--age-band", age_band] + force_flag,
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
    )
    return (cohort_name, age_band, r.returncode, r.stdout, r.stderr)

with ThreadPoolExecutor(max_workers=PARALLEL_WORKERS) as ex:
    futures = {ex.submit(run_bupar_one, c, ab): (c, ab) for c, ab in combinations}
    for fut in as_completed(futures):
        cohort_name, age_band, code, stdout, stderr = fut.result()
        print(f"  [BupaR] {cohort_name} / {age_band} -> exit {code}")
        if code != 0:
            ab_f = age_band.replace("-", "_")
            print(f"    create_bupar_visuals failed (exit {code}). Check 9_dashboard_visuals/logs/bupaR/bupar_{cohort_name}_{ab_f}.log if available")
            if stderr:
                print("    stderr:", (stderr[:1500] + "..." if len(stderr) > 1500 else stderr))
            if stdout:
                print("    stdout:", (stdout[:800] + "..." if len(stdout) > 800 else stdout))
            if FAIL_FAST:
                raise RuntimeError(f"BupaR failed: {cohort_name} / {age_band}")
print("BupaR done.")

## Run FP-Growth (itemsets, Plotly network HTML, S3 upload)

FP-Growth uses **SHAP/FFA-refined** model data: inputs come from `4_model_data` (built from Step 3b `cohort_feature_importance.csv`). **Item types included: drugs (`drug_name`), ICD diagnosis codes (`icd_code`), and CPT procedure codes (`cpt_code`)** (plus combined `medical_code`). For each cohort/age band this step: (1) ensures itemsets exist, (2) creates PNGs and **Plotly interactive network HTML**, (3) **uploads to the dashboard bucket** (e.g. `jerome-dixon.io`) under `{S3_DASHBOARD_PREFIX}/fpgrowth/{cohort}/{age_band}/plots/` (e.g. `vcu/pgx-risk-calculator/fpgrowth/...`). The dashboard then shows the **network plot for the user-selected cohort** via the `/visualizations/fpgrowth` API. **FP-Growth features are not added to model data** (same as DTW); they are for dashboard visualization only.

Run the cell below in parallel (FPGROWTH_WORKERS at a time; builds itemsets, Plotly HTML, uploads to S3). **Exit 0** = itemsets/plots produced for that cohort/age_band; **exit 1** = no outputs (e.g. model_data missing or too few transactions). Logs: `9_dashboard_visuals/logs/fpgrowth/` and S3 `4_fpgrowth_log/{cohort}/{age_band}/`.

In [ ]:
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

try:
    FAIL_FAST
except NameError:
    FAIL_FAST = True
try:
    FPGROWTH_WORKERS
except NameError:
    FPGROWTH_WORKERS = 4
force_flag = ["--force"] if FORCE_RERUN else []

def run_fpgrowth_one(cohort_name, age_band):
    r = subprocess.run(
        [sys.executable, str(FPGROWTH_VISUALS_SCRIPT), "--cohort-name", cohort_name, "--age-band", age_band] + force_flag,
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
    )
    return (cohort_name, age_band, r.returncode, r.stdout, r.stderr)

with ThreadPoolExecutor(max_workers=FPGROWTH_WORKERS) as ex:
    futures = {ex.submit(run_fpgrowth_one, c, ab): (c, ab) for c, ab in combinations}
    for fut in as_completed(futures):
        cohort_name, age_band, code, stdout, stderr = fut.result()
        print(f"  [FP-Growth] {cohort_name} / {age_band} -> exit {code}")
        if code != 0:
            ab_f = age_band.replace("-", "_")
            print(f"    No itemsets produced (exit 1). Check 9_dashboard_visuals/logs/fpgrowth/fpgrowth_{cohort_name}_{ab_f}.log or s3://pgx-repository/4_fpgrowth_log/{cohort_name}/{age_band}/")
            if stderr:
                print("    stderr:", (stderr[:1500] + "..." if len(stderr) > 1500 else stderr))
            if stdout:
                print("    stdout:", (stdout[:800] + "..." if len(stdout) > 800 else stdout))
            if FAIL_FAST:
                raise RuntimeError(f"FP-Growth failed: {cohort_name} / {age_band}")
print("FP-Growth done.")

## Run DTW trajectory analysis (visualization only)

**DTW trajectories are for visualization/exploration only** (not model features). This step:
1. **Extracts lightweight trajectories** from model_data filtered by SHAP/FFA important codes (same as BupaR/FP-Growth)
2. **Creates visualizations** (trajectory cluster plots, routine vs no routine charts)
3. **Uploads to dashboard bucket** under `{S3_DASHBOARD_PREFIX}/dtw/{cohort}/{age_band}/plots/`

**What's created:**
- `dtw_features_{cohort}_{age_band}.csv` - Minimal trajectory data:
  - `seq_pattern_str`: Sequence of activity codes (e.g., "DRUG:Med_ICD:F1120_CPT:99213")
  - `admin_icd_event_count`: Count of administrative ICD codes (routine vs no routine)

  - `trajectory_length`, `trajectory_diversity`: Basic metrics**Runtime:** ~1-2 minutes per cohort/age_band (fast - no expensive DTW distance computations!)

- Trajectory cluster plots (3D for opioid_ed, 1D for polypharmacy)

- `chart_data.json` (routine_comparison, high_risk_trajectories) for dashboard**Research Question:** Addresses "Routine vs no routine appointments → outcomes" via admin ICD analysis. Shows outcome rates for patients with/without routine care patterns. **Date scope:** full pipeline (2016–2019).


In [ ]:
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

try:
    FAIL_FAST
except NameError:
    FAIL_FAST = True  # Stop on first failure; set False to continue (also in config/BupaR cell)
force_flag = ["--force"] if FORCE_RERUN else []

def run_dtw_one(cohort_name, age_band):
    """Run DTW trajectory extraction and visualization (two-step process)."""
    # Step 1: Extract lightweight trajectories from model_data (filtered by SHAP/FFA)
    r_traj = subprocess.run(
        [sys.executable, str(DTW_TRAJECTORIES_SCRIPT), 
         "--cohort", cohort_name, "--age-band", age_band] + force_flag,
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
    )
    
    # Check trajectory extraction success
    if r_traj.returncode != 0:
        return (cohort_name, age_band, r_traj.returncode, r_traj.stdout, r_traj.stderr, None, None)
    
    # Step 2: Create and publish visualizations
    r_vis = subprocess.run(
        [sys.executable, str(DTW_VISUALS_SCRIPT), "--cohort-name", cohort_name, "--age-band", age_band,
         "--project-root", str(REPO_ROOT)] + force_flag,
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
    )
    

    return (cohort_name, age_band, r_traj.returncode, r_traj.stdout, r_traj.stderr, print("DTW done (trajectories + visuals).")

            r_vis.returncode, r_vis.stdout)

                    raise RuntimeError(f"DTW visualization failed: {cohort_name} / {age_band}")

with ThreadPoolExecutor(max_workers=PARALLEL_WORKERS) as ex:                if vis_code != 0 and FAIL_FAST:

    futures = {ex.submit(run_dtw_one, c, ab): (c, ab) for c, ab in combinations}                print(f"    Visuals: exit {vis_code}")

    for fut in as_completed(futures):                vis_code = result[5]

        result = fut.result()            if len(result) > 5 and result[5] is not None:

        cohort_name, age_band = result[0], result[1]            # Trajectory extraction succeeded, check visualization

        traj_code = result[2]        else:

                        raise RuntimeError(f"DTW trajectory extraction failed: {cohort_name} / {age_band}")

        print(f"  [DTW] {cohort_name} / {age_band}")            if FAIL_FAST:

        print(f"    Trajectories: exit {traj_code}")                print("      stderr:", (traj_err[:1500] + "..." if len(traj_err) > 1500 else traj_err))

                    if traj_err:

        if traj_code != 0:            traj_err = result[4] if len(result) > 4 else ""

### Appointments vs no appointments and extreme-density cohorts

**Research question (N1):** Is there a difference in outcomes for patients without routine appointments vs those with routine care? The DTW tab answers this via **Routine vs No Routine (Outcomes)** and **High-Risk vs Low-Risk Trajectories** (see above). These are shown over the **full pipeline (2016–2019)**, not a single year.

**Extreme-density cohorts** are high-utilizer patients (top ~5% by medical_code transaction density) split out so they do not dominate main models (see `docs/Step4_ModelData/README_model_data_and_extreme_split.md`). For **each cohort and age band**, running extract + DTW (and optionally BupaR) for the extreme-density subgroup lets you compare **routine vs no routine** (outcomes and trajectories) in the high-utilizer subgroup and how **extreme densities** and **extreme-density trajectories** differ across age bands and cohorts. By default the cell below uses the **same (cohort, age_band) combinations** as the main pipeline. Set `EXTREME_COMBINATIONS = []` to skip. Requires **Step 4** (model data) first.

In [ ]:
# Default: same (cohort, age_band) as main pipeline so we get routine vs no routine and
# extreme-density trajectories for every cohort and age band. Set to [] to skip.
EXTREME_COMBINATIONS = combinations  # from config cell above

EXTREME_EXTRACT_SCRIPT = STEP9_ROOT / "dtw" / "extract_extreme_density_cohort.py"
extreme_force_flag = ["--force"] if FORCE_RERUN else []

def run_extreme_one(cohort_name, age_band):
    r0 = subprocess.run(
        [sys.executable, str(EXTREME_EXTRACT_SCRIPT), "--cohort-name", cohort_name, "--age-band", age_band],
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
    )
    if r0.returncode != 0:
        return (cohort_name, age_band, r0.returncode, None, r0.stdout, r0.stderr, None, None)
    extreme_name = f"{cohort_name}_extreme_density"
    r2 = subprocess.run(
        [sys.executable, str(DTW_VISUALS_SCRIPT), "--cohort-name", extreme_name, "--age-band", age_band,
         "--project-root", str(REPO_ROOT)] + extreme_force_flag,
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
    )
    return (cohort_name, age_band, r0.returncode, r2.returncode, r0.stdout, r0.stderr, r2.stdout, r2.stderr)

if not EXTREME_COMBINATIONS:
    print("EXTREME_COMBINATIONS is empty; skipping extreme-density cohort extraction and DTW.")
else:
    from concurrent.futures import ThreadPoolExecutor, as_completed
    with ThreadPoolExecutor(max_workers=PARALLEL_WORKERS) as ex:
        futures = {ex.submit(run_extreme_one, c, ab): (c, ab) for c, ab in EXTREME_COMBINATIONS}
        for fut in as_completed(futures):
            result = fut.result()
            cohort_name, age_band = result[0], result[1]
            c0, c2 = result[2], result[3]
            print(f"  [Extreme] {cohort_name} / {age_band} -> extract={c0}, dtw_vis={c2}")
            if c0 != 0:
                print(f"    extract_extreme_density_cohort failed (exit {c0})")
                if len(result) > 5 and result[5]:
                    print("    stderr:", (result[5][:1500] + "..." if len(result[5]) > 1500 else result[5]))
                if FAIL_FAST:
                    raise RuntimeError(f"Extract extreme cohort failed: {cohort_name} / {age_band}")
            if c2 is not None and c2 != 0:
                print(f"    create_dtw_visuals failed (exit {c2})")
                if FAIL_FAST:
                    raise RuntimeError(f"DTW create_dtw_visuals failed: {cohort_name}_extreme_density / {age_band}")
    print(f"Done: extreme-density extract + DTW visuals for {len(EXTREME_COMBINATIONS)} combinations (parallel).")

## View visualization outputs

**Outputs are not in the repo** — they are created when you run the BupaR, FP-Growth, and DTW cells above. Paths: **`10_risk_dashboard/visualizations/bupar/outputs/`**, **`.../dtw/outputs/`**, **`.../fpgrowth/outputs/`** (each has `&lt;cohort&gt;/&lt;age_band&gt;/plots/` with PNG and HTML). Run the cell below **after** those steps to list paths and preview the first PNG (and an "Open in browser" link for HTML). If you see "No outputs yet", run the pipeline cells above first.

In [ ]:
# Where outputs live and a sample preview (run after BupaR / FP-Growth / DTW cells)
from pathlib import Path
from IPython.display import display, Image, HTML, IFrame

def _first_plots_dir(base: Path, subdir: str):
    out = base / subdir / "outputs"
    if not out.exists():
        return None
    for cohort in out.iterdir():
        if not cohort.is_dir() or cohort.name.startswith("."):
            continue
        for age in cohort.iterdir():
            if not age.is_dir():
                continue
            plots = age / "plots"
            if plots.exists():
                return plots
    return None

base = REPO_ROOT / "10_risk_dashboard" / "visualizations"
print("Output root:", base)
print()

for name, subdir in [("BupaR", "bupar"), ("DTW", "dtw"), ("FP-Growth", "fpgrowth")]:
    plots_dir = _first_plots_dir(base, subdir)
    print(f"--- {name} ---")
    if not plots_dir:
        print(f"  No outputs yet. Run the \"Run {name}\" cell above, then re-run this cell to see visuals here.")
        print(f"  Path checked: {base / subdir / 'outputs'}")
        continue
    print(f"  Sample dir: {plots_dir}")
    pngs = sorted(plots_dir.glob("*.png"))
    htmls = sorted(plots_dir.glob("*.html"))
    print(f"  PNGs: {len(pngs)}, HTMLs: {len(htmls)}")
    if pngs:
        display(HTML(f"<b>{name} (first PNG)</b>"))
        display(Image(filename=str(pngs[0]), width=600))
    if htmls:
        display(HTML(f"<b>{name} (first HTML)</b>"))
        try:
            display(IFrame(src=str(htmls[0]), width=700, height=450))
        except Exception as e:
            pass
        # Local file IFrame often blocked; show path to open in browser
        display(HTML(f'<a href="file:///{htmls[0].resolve().as_posix()}" target="_blank">Open in browser</a>'))
    print()
print("All outputs: 10_risk_dashboard/visualizations/{bupar,dtw,fpgrowth}/outputs/<cohort>/<age_band>/plots/")

## API (reference)

Lambda receives **user input** (cohort, age_band, model/feature selections) and **filters** only—it does not process or generate visualization data. All BupaR, DTW, and FP-Growth visuals are **prebuilt on EC2** and **saved to S3**; the API returns **URLs** to those prebuilt assets (filtered by cohort/age_band). Endpoints: `GET /visualizations/causal`, `/visualizations/bupar`, `/visualizations/dtw`, `/visualizations/fpgrowth`. See `10_risk_dashboard/backend/README.md`.

In [ ]:
print("Dashboard endpoints: 10_risk_dashboard/backend/README.md")
print("API Gateway deploy: utility_scripts/create_api_gateway_pgx_risk_calculator.sh")

## Next: Build and deploy

Build and deploy run **only** in [5_build_and_deploy.ipynb](5_build_and_deploy.ipynb). Run that notebook after this one.

In [ ]:
# Build and deploy run only in 5_build_and_deploy.ipynb. Run that notebook after this one.
print("Build and deploy (once): open 5_build_and_deploy.ipynb and run it after this notebook.")

*(Build and deploy — including frontend sync to S3 — are done only in notebook 3. See above.)*